# Where the Dimension Hides — complete analysis pipeline

**Paper:** *Where the Dimension Hides: The Approximation Constant of Kolmogorov–Arnold Networks*
**Author:** Haydar Kılıç · ORCID 0000-0002-2551-3772

This notebook contains the computations of the **pre-registered study** (Sections 3–4 of the
manuscript), in the order the paper presents it. It replaces the original development notebook:
the e-mail alerts, crash hooks, `vmap` seed-parallel trainer and checkpoint-resume machinery
have been removed, because they served the multi-day A100 run rather than the science.

Three companion notebooks in this repository cover the rest:

| Notebook | Covers | Manuscript |
|---|---|---|
| `KAN_followup_experiments.ipynb` | capacity-decoupled sweep, re-specified absorption contrast, first certification attempt | §4.11 |
| `KAN_completion.ipynb` | depth-controlled width probe, real-data pipeline with the repaired budget grid, integrity checks | §4.10, §4.11 |
| `KAN_optional.ipynb` | working certificate for the two-layer family, appended-edge definitions, hidden width decoupled from dimension, replication subset | §4.10, §4.11, §5.2 |

Between the four, every number in the manuscript is reproducible. Run them in that order if
starting from scratch; each is independently resumable.

**Two modes.**

| Mode | What runs | Hardware | Time |
|---|---|---|---|
| `ANALYSIS_ONLY = True` | closed-form probes + every table and figure, from an existing `runs_raw.csv` | CPU | ~5 min |
| `ANALYSIS_ONLY = False` | the full training sweep as well | A100 | days |

Start with `ANALYSIS_ONLY = True`. It reproduces every number in the paper except those that
require re-training.

**Structure**

1. Configuration and pre-registered constants
2. The spline edge and the architectures built from it
3. Target functions
4. The closed-form oracle and the sandwich inequality
5. Premise measurements: bedrock rate, single edge, analytic reduction
6. Controls: smoothness saturation, numerical floor, phase randomisation
7. Design power analysis
8. The training sweep *(skipped when `ANALYSIS_ONLY`)*
9. The joint edge-law regression, in two parametrisations
10. Direct premise test and capacity regime
11. Sandwich, absorption gap, critical dimension, significance
12. Figures
13. Pre-registered verdict

## 1. Configuration and pre-registered constants

Every tolerance below was fixed **before** any model was fitted. Nothing in this notebook
changes them; the verdict in Section 13 applies them verbatim.

In [ ]:
import os, json, math, time, random, hashlib, platform
from dataclasses import dataclass, field, asdict
from typing import List

import numpy as np
import pandas as pd
from scipy import stats

# ---- Where the run log lives --------------------------------------------------------
# Point this at the outputs directory of a completed run. When ANALYSIS_ONLY is True the
# notebook reads runs_raw.csv (or checkpoints/runs.jsonl) from here and re-derives every
# table and figure. When False, the sweep in Section 8 writes it.
# The search below finds the committed run automatically whether the notebook is opened
# from the repository root or from notebooks/. Set OUT_DIR by hand to override.
def _find_outputs():
    for cand in ("results", "../results",
                 "kan_hidden_curse_v5/outputs", "../kan_hidden_curse_v5/outputs"):
        if os.path.exists(os.path.join(cand, "runs_raw.csv")):
            return os.path.abspath(cand)
    return os.path.abspath("kan_hidden_curse_v5/outputs")   # fresh run: created below

OUT_DIR = _find_outputs()
FIG_DIR = os.path.join(OUT_DIR, "figures")
for d in (OUT_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)
print("outputs directory:", OUT_DIR)

ANALYSIS_ONLY = True     # True: no training, CPU only. False: run the full sweep.

# ---- Pre-registered decision constants (Table 3 of the manuscript) -------------------
THRESHOLDS = dict(
    bedrock_rate_rel_tol = 0.15,   # |slope + (k+1)| / (k+1)
    cN_rel_tol           = 0.20,
    cE_rel_tol           = 0.25,
    ka_must_exceed_add   = True,
    min_seeds_for_sig    = 6,
    fdr_alpha            = 0.05,
    converge_mse         = 0.05,
    sandwich_train_min   = 0.95,
)
SAT_DEFAULT = 0.6                  # saturation filter for probe fits
SAT_GRID    = [0.5, 0.6, 0.7]      # sensitivity sweep
SAT_P2      = 0.8                  # looser filter for trained comparison fits
MIN_FIT_POINTS = 4
VIF_MAX     = 10.0
N_BOOT      = 2000

@dataclass
class Config:
    """The experimental axes. These are the `full` mode values used in the paper."""
    spline_orders: List[int] = field(default_factory=lambda: [2, 3, 4])
    dims:          List[int] = field(default_factory=lambda: [2, 4, 8, 16, 32])
    param_budgets: List[int] = field(default_factory=lambda:
                                     [2_000, 4_000, 8_000, 16_000, 32_000, 64_000, 128_000])
    iso_grids:     List[int] = field(default_factory=lambda: [4, 6, 8, 12, 16, 24, 32, 48, 64])
    alpha_grid:  List[float] = field(default_factory=lambda: [0.0, 0.25, 0.5, 0.75, 1.0])
    lr_grid:     List[float] = field(default_factory=lambda: [1e-3, 3e-3, 1e-2])
    seeds:         List[int] = field(default_factory=lambda: list(range(10)))
    bedrock_grids: List[int] = field(default_factory=lambda: [4, 8, 16, 32, 64, 128])
    width_grid:    List[int] = field(default_factory=lambda: [1, 2, 4, 8, 16])
    real_datasets: List[str] = field(default_factory=lambda:
                                     ["airfoil", "concrete", "energy", "wine", "superconduct"])
    n_train: int = 16_000
    n_test:  int = 8_000
    epochs:  int = 500
    batch_size: int = 2_048
    lr: float = 3e-3
    weight_decay: float = 1e-5
    kan_grid_default: int = 8
    mlp_depth: int = 2

CFG = Config()
print("mode:", "analysis only" if ANALYSIS_ONLY else "full sweep")
print(json.dumps(THRESHOLDS, indent=2))

## 2. The spline edge and the architectures built from it

Everything in the study is a composition of one primitive: a univariate B-spline of order $k$
on a uniform grid of $G$ intervals, plus a SiLU residual term.

$$\phi(x) \;=\; \sum_{j} c_{j}\,B_{j,k}(x) \;+\; w\,\mathrm{silu}(x)$$

The basis comes from the Cox–de Boor recursion. The one fact that drives the entire paper is
the exact parameter count (Lemma 1):

$$N \;=\; E\,(G + k + 1),$$

so at a fixed budget the resolution purchasable per edge is $G = N/E - (k+1)$.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.use_deterministic_algorithms(True, warn_only=True)

def seed_everything(s: int) -> None:
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


class KANLayer(nn.Module):
    """B-spline KAN layer: learnable univariate functions on edges, summation at nodes.

    Parameters per edge: (G + k) spline coefficients + 1 residual weight.
    """
    def __init__(self, in_dim, out_dim, grid=8, order=3, grid_range=(-1., 1.)):
        super().__init__()
        self.order, self.grid, self.grid_range = order, grid, grid_range
        g0, g1 = grid_range
        step = (g1 - g0) / grid
        knots = torch.arange(-order, grid + order + 1) * step + g0
        self.register_buffer("knots", knots)
        n_basis = grid + order
        self.coeff = nn.Parameter(torch.randn(in_dim, out_dim, n_basis) * 0.1)
        self.w_res = nn.Parameter(torch.randn(in_dim, out_dim) * 0.1)

    def _basis(self, x):
        """Cox-de Boor recursion. Returns (..., n_basis)."""
        g0, g1 = self.grid_range
        eps = 1e-6 * (g1 - g0)
        x = x.clamp(g0 + eps, g1 - eps).unsqueeze(-1)
        t = self.knots
        b = ((x >= t[:-1]) & (x < t[1:])).to(x.dtype)
        for k in range(1, self.order + 1):
            ld = (t[k:-1]   - t[:-(k + 1)]).clamp_min(1e-8)
            rd = (t[k + 1:] - t[1:-k]     ).clamp_min(1e-8)
            b = (x - t[:-(k + 1)]) / ld * b[..., :-1] + (t[k + 1:] - x) / rd * b[..., 1:]
        return b

    def forward(self, x):
        b = self._basis(x)
        return (torch.einsum("bik,iok->bio", b, self.coeff)
                + F.silu(x).unsqueeze(-1) * self.w_res).sum(1)


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


# ---- The four architectures ---------------------------------------------------------
class EdgeKAN(nn.Module):
    """Additive probe: one layer d -> 1.  E = d."""
    def __init__(self, d, grid, order):
        super().__init__(); self.kan = KANLayer(d, 1, grid=grid, order=order)
    def forward(self, x): return self.kan(x)


class KANet(nn.Module):
    """Two-layer Kolmogorov-Arnold probe, hidden width H = 2d+1.  E = (2d+1)(d+1) ~ 2d^2."""
    def __init__(self, d, grid, order):
        super().__init__()
        H = 2 * d + 1
        self.l1 = KANLayer(d, H, grid=grid, order=order)
        self.l2 = KANLayer(H, 1, grid=grid, order=order)
    def forward(self, x): return self.l2(self.l1(x))


class WidthKAN(nn.Module):
    """Depth-controlled probe: one layer d -> m with a FIXED uniform read-out.  E = m*d.

    Used in the design-power analysis (Section 7) to decorrelate log E from log N.
    It is not part of the fitted sweep; see the manuscript, Section 3.2.
    """
    def __init__(self, d, m, grid, order):
        super().__init__()
        self.kan = KANLayer(d, m, grid=grid, order=order)
        self.register_buffer("mix", torch.ones(m) / m)
    def forward(self, x): return (self.kan(x) * self.mix).sum(1, keepdim=True)


class FullKAN(nn.Module):
    """Budget-matched practical KAN: projection + residual KAN block + head."""
    def __init__(self, d, width, grid, order):
        super().__init__()
        self.inp = nn.Linear(d, width); self.norm = nn.LayerNorm(width)
        self.kan = KANLayer(width, width, grid=grid, order=order)
        self.head = nn.Linear(width, 1)
    def forward(self, x):
        h = self.inp(x)
        return self.head(h + self.kan(self.norm(h)))


class MLP(nn.Module):
    """Parameter-matched baseline: projection + pre-norm residual blocks + head."""
    def __init__(self, d, width, depth=2):
        super().__init__()
        self.inp = nn.Linear(d, width)
        self.blocks = nn.ModuleList([nn.ModuleList(
            [nn.LayerNorm(width), nn.Linear(width, width), nn.Linear(width, width)])
            for _ in range(depth)])
        self.head = nn.Linear(width, 1)
    def forward(self, x):
        h = self.inp(x)
        for ln, a, b in self.blocks:
            h = h + b(F.gelu(a(ln(h))))
        return self.head(h)


def edge_count(kind, d):
    """Number of univariate spline edges. This is the regressor of record."""
    if kind in ("edge_kan", "best_spline"): return d
    if kind == "kanet":                     return (2 * d + 1) * (d + 1)
    return None                             # full_kan / mlp are budget-matched, not edge-matched


def _closest_width(a, b, budget, wmax):
    """Solve a*w^2 + b*w + 1 = budget and pick the best integer width."""
    disc = b * b - 4 * a * (1 - budget)
    w0 = (-b + math.sqrt(max(disc, 0.0))) / (2 * a) if a > 0 else (budget - 1) / max(b, 1e-9)
    cands = {int(math.floor(w0)) - 1, int(math.floor(w0)),
             int(math.ceil(w0)), int(math.ceil(w0)) + 1}
    cands = [w for w in cands if 2 <= w < wmax] or [2]
    return min(cands, key=lambda w: abs(a * w * w + b * w + 1 - budget))

def full_kan_for_budget(d, budget, grid, order):
    return FullKAN(d, _closest_width(grid + order + 1, d + 4, budget, 1024), grid, order)

def mlp_for_budget(d, budget, depth):
    return MLP(d, _closest_width(2 * depth, 4 * depth + d + 2, budget, 2048), depth)


# Sanity: the parameter-count identity of Lemma 1
for d, G, k in [(8, 16, 3), (32, 64, 4)]:
    E = edge_count("edge_kan", d)
    assert count_params(EdgeKAN(d, G, k)) == E * (G + k + 1)
print("Lemma 1 (N = E(G+k+1)) verified on the implementation.")
print("device:", DEVICE)

## 3. Target functions

Each probe is evaluated on a target its own function class can represent, so that the only
source of error is spline resolution. All targets are $C^\infty$ and are scaled by
$d^{-1/2}$ so that their variance stays $O(1)$ as the dimension grows.

In [ ]:
def _standardise(y_tr, y_te):
    mu, sd = y_tr.mean(), y_tr.std().clamp_min(1e-8)
    return (y_tr - mu) / sd, (y_te - mu) / sd


def _additive(X):
    """f = d^{-1/2} sum_p psi(x_p).  Exactly representable by EdgeKAN."""
    psi = lambda t: torch.sin(math.pi * t) + 0.3 * torch.cos(3 * math.pi * t)
    return psi(X).sum(1, keepdim=True) / math.sqrt(X.shape[1])


def _ka_struct(X, seed):
    """Planted Kolmogorov-Arnold target: needs all ~2d^2 edges of KANet."""
    d = X.shape[1]; H = 2 * d + 1
    g = torch.Generator().manual_seed(seed + 31)
    a  = torch.randn(H, d, generator=g)
    ph = torch.rand(H, d, generator=g) * 2 * math.pi
    c  = torch.randn(H, 1, generator=g) / math.sqrt(H)
    inner = (a.unsqueeze(0) * torch.sin(math.pi * X.unsqueeze(1) + ph.unsqueeze(0))).sum(-1)
    return torch.tanh(inner / math.sqrt(d)) @ c


def _ka_struct_nonplanted(X, seed):
    """Non-planted K-A target: Chebyshev inner ridges + softsign outer map.

    Neither map is drawn from the probe's own B-spline/tanh family, so the headline is
    tested off the architecture's own construction.
    """
    d = X.shape[1]; H = 2 * d + 1
    g = torch.Generator().manual_seed(seed + 131)
    freqs = torch.randint(1, 4, (H, d), generator=g).float()
    w_in  = torch.randn(H, d, generator=g) / math.sqrt(d)
    inner = (w_in.unsqueeze(0) *
             torch.cos(freqs.unsqueeze(0) *
                       torch.acos(X.clamp(-0.999, 0.999)).unsqueeze(1))).sum(-1)
    inner = inner / math.sqrt(d)
    w_out = torch.randn(H, 1, generator=g) / math.sqrt(H)
    return (inner / (1.0 + inner.abs())) @ w_out


def _interaction(X, seed):
    """Non-separable component: radial bumps + a genuine product term."""
    d = X.shape[1]
    g = torch.Generator().manual_seed(seed + 71)
    centres = torch.rand(8, d, generator=g) * 2 - 1
    w = torch.randn(8, 1, generator=g)
    d2 = ((X.unsqueeze(1) - centres.unsqueeze(0)) ** 2).sum(-1) / d
    return torch.exp(-1.5 * d2) @ w + 0.5 * X[:, :min(d, 4)].prod(1, keepdim=True)


def make_target(kind, d, n_tr, n_te, seed):
    """Returns (X_tr, y_tr, X_te, y_te) on DEVICE, targets standardised on the train split."""
    g = torch.Generator().manual_seed(seed + 1000)
    Xtr = torch.rand(n_tr, d, generator=g) * 2 - 1
    Xte = torch.rand(n_te, d, generator=g) * 2 - 1
    if kind == "additive":
        fn = _additive
    elif kind == "kastruct":
        fn = lambda X: _ka_struct(X, seed)
    elif kind == "kastruct_np":
        fn = lambda X: _ka_struct_nonplanted(X, seed)
    elif kind.startswith("interp"):
        alpha = float(kind.split("@")[1]) if "@" in kind else 0.5
        fa, fi = _additive, lambda X: _interaction(X, seed)
        sa = fa(Xtr).std().clamp_min(1e-8); si = fi(Xtr).std().clamp_min(1e-8)
        fn = lambda X: (1 - alpha) * fa(X) / sa + alpha * fi(X) / si
    else:
        raise ValueError(f"unknown target {kind!r}")
    ytr, yte = _standardise(fn(Xtr), fn(Xte))
    to = lambda t: t.to(DEVICE, torch.float32)
    return to(Xtr), to(ytr), to(Xte), to(yte)


for kind in ["additive", "kastruct", "kastruct_np", "interp@0.0", "interp@1.0"]:
    stds = [round(make_target(kind, d, 512, 256, 0)[1].std().item(), 2) for d in (2, 8, 32)]
    print(f"{kind:14s} target std at d = 2, 8, 32:  {stds}")

## 4. The closed-form oracle and the sandwich inequality

The additive KAN layer is **linear in its parameters**, so its global training optimum is
available in closed form. This is the instrument the whole study rests on: it gives exponents
that no optimisation failure can contaminate, and it bounds any trained member of the same
family from below (Proposition 2).

In [ ]:
def relative_l2(pred, y):
    return (torch.linalg.norm(pred - y) / torch.linalg.norm(y).clamp_min(1e-12)).item()


def best_spline_error(target, d, n_tr, n_te, seed, grid, order):
    """Global least-squares optimum of the EXACT EdgeKAN family.

    The design matrix spans precisely the probe's hypothesis class (spline basis per
    coordinate + SiLU residual column), so the solution is the global train-MSE optimum
    and trained_train >= this, exactly.

    Returns (train relative L2, test relative L2).
    """
    Xtr, ytr, Xte, yte = make_target(target, d, n_tr, n_te, seed)
    layer = EdgeKAN(d, grid, order)

    def solve(dtype, device):
        m = layer.to(device)
        with torch.no_grad():
            def feats(X):
                X = X.to(device)
                b = m.kan._basis(X).reshape(X.shape[0], -1)
                return torch.cat([b, F.silu(X)], 1).to(dtype)
            A = feats(Xtr)
            sol = torch.linalg.lstsq(A, ytr.to(device).to(dtype)).solution
            return (relative_l2((A @ sol).float(), ytr.to(device).float()),
                    relative_l2((feats(Xte) @ sol).float(), yte.to(device).float()))

    # precision ladder: degrade only when capacity forces it
    try:
        return solve(torch.float64, DEVICE)
    except RuntimeError:
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        try:
            return solve(torch.float32, DEVICE)
        except RuntimeError:
            return solve(torch.float64, torch.device("cpu"))


tr, te = best_spline_error("additive", 8, 4000, 2000, 0, 16, 3)
print(f"oracle at d=8, G=16, k=3:  train rel-L2 = {tr:.2e}   test rel-L2 = {te:.2e}")

## 5. Premise measurements

Three measurements that the rest of the paper depends on:

* **B0**, the bedrock one-dimensional rate — pure spline theory, and a calibration of the
  whole apparatus;
* the same rate on a **single isolated outer edge** of the two-layer architecture, which is
  the premise of the analytic reduction;
* the **edge-count elasticity** $\mathrm{d}\log E/\mathrm{d}\log d$ that converts that rate
  into a dimensional exponent.

In [ ]:
def bedrock_rate(cfg, seeds=(0, 1, 2)):
    """B0: closed-form 1-D fit, slope of log(error) on log(G) must be -(k+1)."""
    rows = []
    for k in cfg.spline_orders:
        errs = []
        for G in cfg.bedrock_grids:
            e = np.mean([best_spline_error("additive", 1, cfg.n_train, cfg.n_test, s, G, k)[1]
                         for s in seeds])
            errs.append(e)
        good = [(G, e) for G, e in zip(cfg.bedrock_grids, errs) if e > 0]
        slope = np.polyfit(np.log([g for g, _ in good]), np.log([e for _, e in good]), 1)[0]
        rows.append(dict(k=int(k), measured_slope=round(float(slope), 3), predicted=-(k + 1),
                         passes=bool(abs(slope + (k + 1))
                                     <= THRESHOLDS["bedrock_rate_rel_tol"] * (k + 1))))
    return pd.DataFrame(rows)


def single_edge_rate(cfg, grids=(4, 8, 16, 32, 64)):
    """One isolated edge of the K-A outer layer is a 1-D spline; it must obey the same rate."""
    rows = []
    for k in cfg.spline_orders:
        errs = []
        for G in grids:
            g = torch.Generator().manual_seed(7)
            x = torch.rand(cfg.n_train, 1, generator=g) * 2 - 1
            y = torch.sin(1.7 * math.pi * x) + 0.3 * torch.cos(2.3 * math.pi * x)
            y = (y - y.mean()) / y.std()
            lay = KANLayer(1, 1, grid=G, order=k)
            with torch.no_grad():
                b = lay._basis(x).reshape(x.shape[0], -1)
                A = torch.cat([b, F.silu(x)], 1).double()
                sol = torch.linalg.lstsq(A, y.double(), rcond=None).solution
                errs.append(relative_l2((A @ sol).float(), y))
        good = [(G, e) for G, e in zip(grids, errs) if e > 1e-7]
        slope = np.polyfit(np.log([g for g, _ in good]), np.log([e for _, e in good]), 1)[0]
        rows.append(dict(k=int(k), single_edge_slope=round(float(slope), 3), predicted=-(k + 1),
                         passes=bool(abs(slope + (k + 1))
                                     <= THRESHOLDS["bedrock_rate_rel_tol"] * (k + 1))))
    return pd.DataFrame(rows)


def edge_count_elasticity(cfg):
    """d log E / d log d for the two-layer topology, over the swept dimensions."""
    dims = np.array(sorted(cfg.dims), float)
    E = np.array([(2 * d + 1) * (d + 1) for d in dims], float)
    beta = float(np.polyfit(np.log(dims), np.log(E), 1)[0])
    return pd.DataFrame([dict(k=int(k), dlogE_dlogd=round(beta, 3),
                              finite_d_slope=round((k + 1) * beta, 2),
                              asymptotic_slope=2 * (k + 1))
                         for k in cfg.spline_orders]), beta


BEDROCK = bedrock_rate(CFG)
SINGLE_EDGE = single_edge_rate(CFG)
REDUCTION, DLOGE = edge_count_elasticity(CFG)

print("B0 bedrock rate (Table 1, left):");            print(BEDROCK.to_string(index=False))
print("\nSingle-edge rate (Table 1, middle):");       print(SINGLE_EDGE.to_string(index=False))
print("\nAnalytic reduction (Table 1, right):");      print(REDUCTION.to_string(index=False))

## 6. Controls

**C1 smoothness saturation** — why the rate is $G^{-(k+1)}$ and not faster on $C^\infty$
targets. **C2 numerical floor** — where the power law ends. **C8 phase randomisation** — a
control added after the fact, testing whether knot–target commensurability explains the
exponent overshoot. It does not; see Section 9 and the manuscript, Section 5.2.

In [ ]:
def _weierstrass(x, H, a=3.0, J=18):
    """Weierstrass-type target of prescribed global Holder exponent H."""
    out = torch.zeros_like(x)
    for j in range(J):
        out = out + (a ** (-j * H)) * torch.cos((a ** j) * math.pi * x)
    return out


def _closed_form_1d(y_fn, G, k, n, seed, dtype=torch.float64):
    g = torch.Generator().manual_seed(seed)
    x = torch.rand(n, 1, generator=g, dtype=torch.float64) * 2 - 1
    y = y_fn(x, seed)
    y = (y - y.mean()) / y.std().clamp_min(1e-12)
    lay = KANLayer(1, 1, grid=G, order=k)
    with torch.no_grad():
        b = lay._basis(x).reshape(n, -1)
        A = torch.cat([b, F.silu(x)], 1).to(dtype)
        sol = torch.linalg.lstsq(A, y.to(dtype), rcond=None).solution
        return relative_l2((A @ sol).float(), y.float())


def smoothness_saturation(cfg, H_values=(0.5, 1.0, 1.5, 2.5, 3.5, 5.0),
                          grids=(8, 16, 32, 64), seeds=(0, 1, 2), floor=1e-5):
    """C1: the rate must follow -min(k+1, H) -- the smaller of order and target smoothness."""
    rows = []
    for H in H_values:
        for k in cfg.spline_orders:
            errs = [(G, float(np.mean([_closed_form_1d(lambda x, s: _weierstrass(x, float(H)),
                                                       G, k, cfg.n_train, s) for s in seeds])))
                    for G in grids]
            good = [(G, e) for G, e in errs if e > floor]
            slope = (np.polyfit(np.log([g for g, _ in good]), np.log([e for _, e in good]), 1)[0]
                     if len(good) >= 2 else float("nan"))
            pred = -min(k + 1, H)
            binds = "order" if (k + 1) < H else ("target" if (k + 1) > H else "tie")
            rows.append(dict(holder_H=float(H), order_k=int(k),
                             measured_slope=round(float(slope), 2),
                             predicted=round(float(pred), 2), binds=binds))
    return pd.DataFrame(rows)


def floor_robustness(cfg, grids=(8, 16, 32, 64, 128, 256)):
    """C2: locate the precision floor and show the bedrock slope is window-robust above it."""
    target = lambda x, s: torch.sin(1.7 * math.pi * x) + 0.3 * torch.cos(2.3 * math.pi * x)
    rows = []
    for k in cfg.spline_orders:
        e32 = [_closed_form_1d(target, G, k, cfg.n_train, 0, torch.float32) for G in grids]
        e64 = [_closed_form_1d(target, G, k, cfg.n_train, 0, torch.float64) for G in grids]
        floor = float(min(e32)); margin = 10 * floor
        windows = {
            "above_floor": [(G, e) for G, e in zip(grids, e32) if e > margin],
            "mid":         [(G, e) for G, e in zip(grids, e32) if e > margin and G <= 64],
            "incl_floor":  [(G, e) for G, e in zip(grids, e32) if e > 0],
        }
        sl = {nm: (np.polyfit(np.log([g for g, _ in w]), np.log([e for _, e in w]), 1)[0]
                   if len(w) >= 2 else float("nan")) for nm, w in windows.items()}
        spread = float(abs(sl["above_floor"] - sl["mid"]))
        rows.append(dict(k=int(k), fp32_floor=f"{floor:.1e}", fp64_floor=f"{min(e64):.1e}",
                         slope_above_floor=round(float(sl["above_floor"]), 2),
                         slope_mid=round(float(sl["mid"]), 2),
                         slope_incl_floor=round(float(sl["incl_floor"]), 2),
                         predicted=-(k + 1), window_spread=round(spread, 3)))
    return pd.DataFrame(rows)


def phase_control(cfg, grids=(4, 6, 8, 12, 16, 24, 32, 48, 64), seeds=range(10), floor=1e-6):
    """C8: does knot-target commensurability explain the exponent overshoot?

    `fixed`  = the paper's target, identical across seeds  -> non-monotone error in G
    `random` = frequencies and phases drawn per seed       -> monotone

    If randomisation also moved the fitted slope towards -(k+1), commensurability would
    explain the overshoot. It does not.
    """
    def fixed(x, s):
        return torch.sin(math.pi * x) + 0.3 * torch.cos(3 * math.pi * x)

    def random_phase(x, s):
        g = torch.Generator().manual_seed(9000 + s)
        r = lambda: torch.rand(1, generator=g, dtype=torch.float64).item()
        f1, f2 = 0.7 + 1.6 * r(), 2.1 + 1.8 * r()
        p1, p2 = 2 * math.pi * r(), 2 * math.pi * r()
        return (torch.sin(math.pi * f1 * x + p1) + 0.3 * torch.cos(math.pi * f2 * x + p2))

    rows = []
    for name, fn in [("fixed (paper)", fixed), ("phase-randomised", random_phase)]:
        for k in cfg.spline_orders:
            curves = [[_closed_form_1d(fn, G, k, cfg.n_train, s) for G in grids] for s in seeds]
            mean_curve = np.mean(curves, axis=0)
            good = [(G, e) for G, e in zip(grids, mean_curve) if e > floor]
            slope = np.polyfit(np.log([g for g, _ in good]), np.log([e for _, e in good]), 1)[0]
            rows.append(dict(target=name, k=int(k), slope=round(float(slope), 3),
                             predicted=-(k + 1),
                             abs_error=round(abs(slope + (k + 1)), 3),
                             monotone_frac=round(float((np.diff(mean_curve) < 0).mean()), 2)))
    return pd.DataFrame(rows)


SMOOTH_SAT = smoothness_saturation(CFG)
FLOOR = floor_robustness(CFG)
PHASE = phase_control(CFG)

print("C1 smoothness saturation (Table 2):")
print(SMOOTH_SAT.pivot(index="order_k", columns="holder_H", values="measured_slope").to_string())
print("\nC2 floor robustness (Table 3):"); print(FLOOR.to_string(index=False))
print("\nC8 phase-randomisation control:"); print(PHASE.to_string(index=False))

## 7. Design power analysis

Before any compute is spent, we ask whether the design *could* recover the law if it were
true. A known edge law is planted on the exact cells the sweep will produce, and we check
whether the joint fit recovers $c_E = k+1$. If it cannot, no real fit on that grid is
trustworthy either.

This is also where `WidthKAN` earns its place: sweeping its output width varies $E$ at fixed
grid, decorrelating $\log E$ from $\log N$.

In [ ]:
def _design_rows(cfg, kinds=("edge_kan", "kanet"), width_grid=None):
    rows = []
    for kind in kinds:
        for d in cfg.dims:
            E = edge_count(kind, d)
            for k in cfg.spline_orders:
                for G in cfg.iso_grids:
                    rows.append(dict(kind=kind, d=d, k=k, grid=G,
                                     edges=E, params=E * (G + k + 1)))
    if width_grid:                                    # add the depth-1 width probe
        for d in cfg.dims:
            for k in cfg.spline_orders:
                for G in cfg.iso_grids:
                    for m in width_grid:
                        E = m * d
                        rows.append(dict(kind="widthkan", d=d, k=k, grid=G,
                                         edges=E, params=E * (G + k + 1)))
    return pd.DataFrame(rows)


def vif(a, b):
    """Variance inflation factor of a on b."""
    B = np.column_stack([np.ones(len(b)), b])
    r = a - B @ np.linalg.lstsq(B, a, rcond=None)[0]
    ss = ((a - a.mean()) ** 2).sum()
    r2 = 1 - (r ** 2).sum() / ss if ss > 0 else 0.0
    return float(1.0 / max(1e-9, 1 - r2))


def design_power(cfg, sigma=0.15, n_boot=2000, seed=0, width_grid=None):
    rng = np.random.default_rng(seed)
    out = []
    for k, g in _design_rows(cfg, width_grid=width_grid).groupby("k"):
        P, E = g["params"].values.astype(float), g["edges"].values.astype(float)
        lP, lE = np.log(P), np.log(E)
        X = np.column_stack([np.ones(len(P)), lP, lE])
        clean = 0.3 - (k + 1) * lP + (k + 1) * lE          # planted ground truth
        draws = np.array([np.linalg.lstsq(X, clean + rng.normal(0, sigma, len(P)),
                                          rcond=None)[0] for _ in range(n_boot)])
        lo, hi = np.percentile(draws[:, 2], [2.5, 97.5])
        tol = THRESHOLDS["cE_rel_tol"] * (k + 1)
        out.append(dict(k=int(k), n_cells=len(P), cond=round(float(np.linalg.cond(X)), 1),
                        vif=round(vif(lP, lE), 2),
                        cE_recovered=round(float(draws[:, 2].mean()), 3),
                        cE_lo=round(float(lo), 3), cE_hi=round(float(hi), 3),
                        cE_ci_width=round(float(hi - lo), 3), tol=round(tol, 3),
                        identifiable=bool(abs(draws[:, 2].mean() - (k + 1)) <= tol
                                          and (hi - lo) <= 2 * tol)))
    return pd.DataFrame(out)


DESIGN_POWER = design_power(CFG)
DESIGN_POWER_WIDTH = design_power(CFG, width_grid=CFG.width_grid)
print("Design power, base design (Table 4a):");       print(DESIGN_POWER.to_string(index=False))
print("\nWith the depth-1 width probe (Table 4b):");  print(DESIGN_POWER_WIDTH.to_string(index=False))
print(f"\nmax VIF  {DESIGN_POWER.vif.max():.2f} -> {DESIGN_POWER_WIDTH.vif.max():.2f}")

## 8. The training sweep

Skipped when `ANALYSIS_ONLY = True`. This is the only part that needs an accelerator: it
trains every cell of the factorial design and appends the results to `runs_raw.csv`.

The trainer is deliberately plain — AdamW, cosine schedule, no early stopping, no gradient
clipping, no grid-extension curriculum. That restriction is a finding of the paper, not an
oversight: see Section 5.2 of the manuscript on the optimisation deficit it exposes.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def train_eval(model, data, cfg, seed, lr=None):
    """Train one model and return train/test relative L2 and the final training MSE."""
    seed_everything(seed)
    Xtr, ytr, Xte, yte = data
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr or cfg.lr, weight_decay=cfg.weight_decay)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    loss_fn = nn.MSELoss()
    gen = torch.Generator().manual_seed(seed)
    dl = DataLoader(TensorDataset(Xtr, ytr), batch_size=cfg.batch_size,
                    shuffle=True, generator=gen)
    model.train()
    last = float("nan")
    for _ in range(cfg.epochs):
        run, nb = 0.0, 0
        for xb, yb in dl:
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward(); opt.step()
            run += loss.item(); nb += 1
        sch.step(); last = run / max(nb, 1)
    model.eval()
    with torch.no_grad():
        return dict(rel_l2=relative_l2(model(Xte).float(), yte),
                    rel_l2_train=relative_l2(model(Xtr).float(), ytr),
                    train_mse=last,
                    converged=bool(last == last and last < THRESHOLDS["converge_mse"]))


def run_sweep(cfg, out_csv):
    """The full factorial sweep. Resumable: completed cells are skipped on re-entry."""
    done = set()
    if os.path.exists(out_csv):
        done = set(pd.read_csv(out_csv, low_memory=False)["key"])
        print(f"resuming: {len(done):,} cells already present")

    jobs = []
    for k in cfg.spline_orders:
        for d in cfg.dims:
            for s in cfg.seeds:
                for G in cfg.iso_grids:                      # probe regime
                    jobs += [("edge_kan", "additive",    k, d, None, G, None, s, cfg.lr),
                             ("best_spline", "additive", k, d, None, G, None, s, cfg.lr),
                             ("kanet", "kastruct",       k, d, None, G, None, s, cfg.lr),
                             ("kanet", "kastruct_np",    k, d, None, G, None, s, cfg.lr)]
                for N in cfg.param_budgets:                  # comparison regime
                    for al in cfg.alpha_grid:
                        for lr in cfg.lr_grid:
                            jobs.append(("full_kan", f"interp@{al}", k, d, N, None, al, s, lr))
                            if k == cfg.spline_orders[0]:
                                jobs.append(("mlp", f"interp@{al}", 0, d, N, None, al, s, lr))

    rows, t0 = [], time.time()
    for i, (kind, tgt, k, d, N, G, al, s, lr) in enumerate(jobs):
        key = f"{kind}|{tgt}|k{k}|d{d}|N{N}|G{G}|a{al}|s{s}|lr{lr}"
        if key in done:
            continue
        E = edge_count(kind, d)
        if kind == "best_spline":
            rel_tr, rel = best_spline_error(tgt, d, cfg.n_train, cfg.n_test, s, G, k)
            p, mse, conv = count_params(EdgeKAN(d, G, k)), None, None
        else:
            data = make_target(tgt, d, cfg.n_train, cfg.n_test, s)
            seed_everything(s)
            if kind == "edge_kan":   m = EdgeKAN(d, G, k)
            elif kind == "kanet":    m = KANet(d, G, k)
            elif kind == "full_kan": m = full_kan_for_budget(d, N, cfg.kan_grid_default, k)
            else:                    m = mlp_for_budget(d, N, cfg.mlp_depth)
            p = count_params(m)
            r = train_eval(m, data, cfg, s, lr=lr)
            rel, rel_tr, mse, conv = (r["rel_l2"], r["rel_l2_train"],
                                      r["train_mse"], r["converged"])
        rows.append(dict(key=key, kind=kind, target=tgt,
                         regime="synthetic", k=k, d=d, N=N, grid=G, alpha=al,
                         seed=s, lr=lr, params=p, edges=E,
                         rel_l2=rel, rel_l2_train=rel_tr, train_mse=mse, converged=conv))
        if len(rows) % 200 == 0:
            pd.DataFrame(rows).to_csv(out_csv, mode="a", header=not os.path.exists(out_csv),
                                      index=False)
            rows = []
            print(f"  {i+1}/{len(jobs)} cells, {(time.time()-t0)/60:.1f} min elapsed")
    if rows:
        pd.DataFrame(rows).to_csv(out_csv, mode="a", header=not os.path.exists(out_csv),
                                  index=False)
    print("sweep complete")


RUNS_CSV = os.path.join(OUT_DIR, "runs_raw.csv")
if ANALYSIS_ONLY:
    print("ANALYSIS_ONLY is True -- skipping the sweep.")
    print("Expecting an existing run log at:", RUNS_CSV)
else:
    run_sweep(CFG, RUNS_CSV)

## 9. The joint edge-law regression, in two parametrisations

The edge law predicts $\varepsilon \asymp C_0 (E/N)^{k+1}$. Fitting

$$\log\varepsilon = a + c_N \log N + c_E \log E \qquad\text{(A, asymptotic)}$$

is what the paper pre-registered — but it inherits the large-$G$ approximation
$G \asymp N/E$. Since $N = E(G+k+1)$ exactly, (A) estimates the exponent with respect to
$G+k+1$, not $G$. We therefore also fit

$$\log\varepsilon = a' + c_G \log G^{\mathrm{exact}}, \qquad G^{\mathrm{exact}} = N/E - (k+1)
\qquad\text{(B, exact)}$$

and report both. **This correction was not pre-registered**; it was found when the
dimensional fit disagreed with the bedrock measurement on the same family. The pre-registered
tolerances are applied to it unchanged.

The nested check asks whether $\log E$ carries information beyond $\log G^{\mathrm{exact}}$.
If it does not, the law's functional form is confirmed.

In [ ]:
PROBE_KINDS = ("edge_kan", "kanet", "best_spline")
SYNTH_TARGETS = ("additive", "kastruct", "kastruct_np")

runs = pd.read_csv(RUNS_CSV, low_memory=False)
probe_raw = runs[runs.kind.isin(PROBE_KINDS)].copy()
print(f"loaded {len(runs):,} cells, of which {len(probe_raw):,} are probe cells")


def aggregate(raw, sat):
    ag = (raw.groupby(["d", "grid", "params", "edges"], dropna=False)["rel_l2"]
             .mean().reset_index().rename(columns={"rel_l2": "err"}))
    return ag[(ag.err < sat) & (ag.err > 0) & (ag.edges > 0) & (ag.params > 0)]


def fit_both(cells, k):
    """Parametrisation (A) and (B) on one filtered cell frame."""
    N = cells["params"].values.astype(float)
    E = cells["edges"].values.astype(float)
    y = np.log(cells["err"].values)
    XA = np.column_stack([np.ones(len(N)), np.log(N), np.log(E)])
    cA = np.linalg.lstsq(XA, y, rcond=None)[0]
    Gx = N / E - (k + 1)
    ok = Gx > 0
    cG = (np.linalg.lstsq(np.column_stack([np.ones(ok.sum()), np.log(Gx[ok])]),
                          y[ok], rcond=None)[0][1]
          if ok.sum() >= MIN_FIT_POINTS else np.nan)
    return dict(cN=float(cA[1]), cE=float(cA[2]), cG=float(cG),
                vif=vif(np.log(N), np.log(E)), n_B=int(ok.sum()))


def joint_fits(probe_raw, sat=SAT_DEFAULT, n_boot=N_BOOT, seed=0):
    """Point estimates, cell-level bootstrap and seed-level cluster bootstrap."""
    rng = np.random.default_rng(seed)
    rows, nested = [], []
    for (kind, target), gt in probe_raw.groupby(["kind", "target"]):
        if target not in SYNTH_TARGETS:
            continue
        for k, graw in gt.groupby("k"):
            cells = aggregate(graw, sat)
            if len(cells) < MIN_FIT_POINTS or cells["d"].nunique() < 2:
                continue
            pt = fit_both(cells, k)

            bE, bG = [], []
            for _ in range(n_boot):                       # cell-level bootstrap
                f = fit_both(cells.iloc[rng.integers(0, len(cells), len(cells))], k)
                bE.append(f["cE"])
                if np.isfinite(f["cG"]): bG.append(f["cG"])

            seeds = np.array(sorted(graw["seed"].dropna().unique()))
            by_seed = {s: graw[graw.seed == s] for s in seeds}
            cE_, cG_ = [], []
            for _ in range(n_boot):                       # seed-level cluster bootstrap
                pick = rng.choice(seeds, size=len(seeds), replace=True)
                agb = aggregate(pd.concat([by_seed[s] for s in pick], ignore_index=True), sat)
                if len(agb) < MIN_FIT_POINTS or agb["d"].nunique() < 2:
                    continue
                f = fit_both(agb, k)
                cE_.append(f["cE"])
                if np.isfinite(f["cG"]): cG_.append(f["cG"])

            q = lambda v: np.percentile(v, [2.5, 97.5]) if len(v) else [np.nan, np.nan]
            qE, qG, kE, kG = q(bE), q(bG), q(cE_), q(cG_)
            rows.append(dict(
                family=f"{kind}/{target}", k=int(k), n_cells=len(cells),
                cN=round(pt["cN"], 3), cE=round(pt["cE"], 3),
                cE_lo=round(qE[0], 3), cE_hi=round(qE[1], 3),
                cE_lo_clu=round(kE[0], 3), cE_hi_clu=round(kE[1], 3),
                cE_pass=bool(abs(pt["cE"] - (k + 1)) <= THRESHOLDS["cE_rel_tol"] * (k + 1)),
                cG=round(pt["cG"], 3), cG_lo=round(qG[0], 3), cG_hi=round(qG[1], 3),
                cG_lo_clu=round(kG[0], 3), cG_hi_clu=round(kG[1], 3),
                cG_pass=bool(abs(pt["cG"] + (k + 1))
                             <= THRESHOLDS["bedrock_rate_rel_tol"] * (k + 1)),
                predicted_cE=k + 1, predicted_cG=-(k + 1),
                vif=round(pt["vif"], 2), identifiable=bool(pt["vif"] < VIF_MAX)))

            # nested check: does log E add anything beyond log G_exact?
            N = cells["params"].values.astype(float); E = cells["edges"].values.astype(float)
            y = np.log(cells["err"].values); Gx = N / E - (k + 1); ok = Gx > 0
            if ok.sum() >= MIN_FIT_POINTS + 1:
                X1 = np.column_stack([np.ones(ok.sum()), np.log(Gx[ok])])
                X2 = np.column_stack([X1, np.log(E[ok])])
                r1 = y[ok] - X1 @ np.linalg.lstsq(X1, y[ok], rcond=None)[0]
                r2 = y[ok] - X2 @ np.linalg.lstsq(X2, y[ok], rcond=None)[0]
                rss1, rss2 = float((r1 ** 2).sum()), float((r2 ** 2).sum())
                dof = ok.sum() - 3
                nested.append(dict(family=f"{kind}/{target}", k=int(k),
                                   rss_G_only=round(rss1, 4), rss_G_plus_E=round(rss2, 4),
                                   F=round(float((rss1 - rss2) / (rss2 / dof)), 2),
                                   coef_on_logE=round(float(
                                       np.linalg.lstsq(X2, y[ok], rcond=None)[0][2]), 3)))
    return pd.DataFrame(rows), pd.DataFrame(nested)


JOINT, NESTED = joint_fits(probe_raw)
print("\nJoint fit, both parametrisations (Tables 7 and 8):")
print(JOINT[["family", "k", "cE", "cE_lo", "cE_hi", "cE_pass",
             "cG", "cG_lo", "cG_hi", "cG_pass", "vif"]].to_string(index=False))
print("\nNested check -- does log E add anything beyond log G_exact?")
print(NESTED.to_string(index=False))

SAT_SENS = pd.concat([joint_fits(probe_raw, sat=s, n_boot=0)[0].assign(sat=s)
                      for s in SAT_GRID])
print("\nSaturation sensitivity (exponents must be stable across the filter):")
print(SAT_SENS.pivot_table(index=["family", "k"], columns="sat",
                           values=["cE", "cG"]).round(3).to_string())

## 10. Direct test of the premise, and the capacity regime

The premise underlying the edge law can be tested **without any regression**: holding the grid
fixed holds per-edge resolution fixed, so a model whose error is governed by resolution alone
must show a dimensional slope of zero at every grid.

The capacity-to-sample ratio $N/n_{\mathrm{tr}}$ then shows which cells are approximation-
limited at all. This is where the two probes turn out not to be comparable.

In [ ]:
def dimensional_slope_at_fixed_grid(probe_raw, k=3):
    """s_d(G): slope of log(error) on log(d) at each fixed grid. Premise predicts 0."""
    rows = {}
    for (kind, target), g in probe_raw.groupby(["kind", "target"]):
        if target not in SYNTH_TARGETS:
            continue
        sub = g[g.k == k]
        ag = sub.groupby(["grid", "d"])["rel_l2"].mean().reset_index()
        out = {}
        for G, gg in ag.groupby("grid"):
            gg = gg.sort_values("d")
            if len(gg) >= 3 and (gg.rel_l2 > 0).all():
                out[int(G)] = round(float(np.polyfit(np.log(gg.d),
                                                     np.log(gg.rel_l2), 1)[0]), 2)
        rows[f"{kind}/{target}"] = out
    return pd.DataFrame(rows).T.sort_index()


def capacity_ratio(cfg, kind, k=3):
    """N / n_train per cell. Values above 1 are over-parametrised."""
    return pd.DataFrame(
        {G: {d: round(edge_count(kind, d) * (G + k + 1) / cfg.n_train, 2) for d in cfg.dims}
         for G in cfg.iso_grids})


DSLOPE = dimensional_slope_at_fixed_grid(probe_raw)
print("Dimensional slope at fixed resolution (Table 9) -- premise predicts 0 everywhere:")
print(DSLOPE.to_string())

print("\nCapacity-to-sample ratio, KANet (Table 10a); bold in the paper means > 1:")
print(capacity_ratio(CFG, "kanet").to_string())
print("\nSame for EdgeKAN (Table 10b):")
print(capacity_ratio(CFG, "edge_kan").to_string())

kn = probe_raw[(probe_raw.kind == "kanet") & (probe_raw.target == "kastruct")
               & (probe_raw.k == 3)]
print("\nKANet train vs test error by dimension (the generalisation gap):")
print(kn.groupby("d")[["rel_l2_train", "rel_l2", "train_mse"]].mean().round(4).to_string())

## 11. The four remaining predictions

**P4** the sandwich · **P2** structural absorption · **P3** critical dimension ·
and the paired KAN-versus-MLP comparison with multiplicity control.

In [ ]:
def sandwich(runs):
    """P4: the trained model can never beat the closed-form optimum on the training split."""
    base = runs[(runs.kind.isin(["edge_kan", "best_spline"])) & (runs.target == "additive")]
    rows = []
    for (k, d, G), _ in base.groupby(["k", "d", "grid"]):
        tr = base[(base.kind == "edge_kan")    & (base.k == k) & (base.d == d) & (base.grid == G)]
        bs = base[(base.kind == "best_spline") & (base.k == k) & (base.d == d) & (base.grid == G)]
        if not len(tr) or not len(bs):
            continue
        rows.append(dict(k=int(k), d=int(d), grid=int(G),
                         lsq_train=bs.rel_l2_train.mean(), trained_train=tr.rel_l2_train.mean(),
                         train_ok=bool(tr.rel_l2_train.mean() >= bs.rel_l2_train.mean() - 1e-3),
                         opt_gap_test=tr.rel_l2.mean() - bs.rel_l2.mean()))
    return pd.DataFrame(rows)


def absorption_gap(runs, cfg):
    """P2: gap = 2(k+1) - measured d-slope, per interaction fraction alpha."""
    fm = runs[runs.kind.isin(["full_kan", "mlp"])]
    best = (fm.groupby(["kind", "target", "alpha", "k", "d", "N", "seed"], dropna=False)
              ["rel_l2"].min().reset_index())
    agg = (best.groupby(["kind", "target", "alpha", "k", "d", "N"], dropna=False)
                ["rel_l2"].mean().reset_index())
    rows = []
    for (al, k, N), g in agg[agg.kind == "full_kan"].groupby(["alpha", "k", "N"], dropna=False):
        if pd.isna(k) or k == 0:
            continue
        g = g[g.rel_l2 < SAT_P2].sort_values("d")
        if g["d"].nunique() < MIN_FIT_POINTS:
            continue
        slope = np.polyfit(np.log(g.d), np.log(g.rel_l2), 1)[0]
        rows.append(dict(alpha=float(al), k=int(k), N=int(N),
                         measured_d_slope=round(float(slope), 3),
                         absorption_gap=round(2 * (k + 1) - float(slope), 3)))
    p2 = pd.DataFrame(rows)
    rho, p = stats.spearmanr(p2.alpha, p2.absorption_gap)
    rng = np.random.default_rng(0)
    slopes = [np.polyfit(p2.alpha.values[i], p2.absorption_gap.values[i], 1)[0]
              for i in (rng.integers(0, len(p2), len(p2)) for _ in range(N_BOOT))
              if len(np.unique(p2.alpha.values[i])) >= 2]
    lo, hi = np.percentile(slopes, [2.5, 97.5])
    trend = dict(spearman_rho=round(float(rho), 3), spearman_p=round(float(p), 4),
                 slope_lo=round(float(lo), 3), slope_hi=round(float(hi), 3),
                 supported=bool(rho < 0 and p < 0.05 and hi < 0))
    return p2, trend, agg


def critical_dimension(agg, cfg):
    """P3: the dimension at which the parameter-matched MLP overtakes the best KAN."""
    rows = []
    for N in cfg.param_budgets:
        for al in cfg.alpha_grid:
            def curve(kind):
                g = agg[(agg.kind == kind) & (agg.alpha == al) & (agg.N == N)]
                if not len(g):
                    return np.array([]), np.array([])
                gg = g.groupby("d")["rel_l2"].min().reset_index().sort_values("d")
                return gg.d.values, gg.rel_l2.values
            dk, ek = curve("full_kan"); dm, em = curve("mlp")
            common = np.intersect1d(dk, dm)
            if len(common) < 2:
                continue
            diff = np.interp(common, dk, ek) - np.interp(common, dm, em)
            dstar = None
            for i in range(len(common) - 1):
                if diff[i] <= 0 < diff[i + 1]:
                    x0, x1, y0, y1 = common[i], common[i+1], diff[i], diff[i+1]
                    dstar = float(x0 + (0 - y0) * (x1 - x0) / (y1 - y0)); break
            rows.append(dict(N=int(N), alpha=float(al),
                             d_star=("never_loses" if dstar is None and diff[-1] <= 0
                                     else "always_loses" if dstar is None else round(dstar, 2))))
    return pd.DataFrame(rows)


def significance(runs):
    """Paired Wilcoxon per cell, rank-biserial effect size, Benjamini-Hochberg at 5%."""
    def rank_biserial(a, b):
        diff = a - b; diff = diff[diff != 0]
        if not len(diff): return 0.0
        r = stats.rankdata(np.abs(diff))
        Rp, Rn = r[diff > 0].sum(), r[diff < 0].sum()
        return float((Rp - Rn) / (Rp + Rn)) if (Rp + Rn) > 0 else 0.0

    eff = (runs[runs.kind.isin(["full_kan", "mlp"])]
           .groupby(["kind", "target", "d", "N", "seed"], dropna=False)["rel_l2"]
           .min().reset_index())
    rows = []
    for (tgt, d, N), _ in eff.groupby(["target", "d", "N"], dropna=False):
        ks = eff[(eff.kind == "full_kan") & (eff.target == tgt) &
                 (eff.d == d) & (eff.N == N)].groupby("seed")["rel_l2"].min()
        ms = eff[(eff.kind == "mlp") & (eff.target == tgt) &
                 (eff.d == d) & (eff.N == N)].groupby("seed")["rel_l2"].min()
        common = ks.index.intersection(ms.index)
        if len(common) < THRESHOLDS["min_seeds_for_sig"]:
            continue
        a, b = ks.loc[common].values, ms.loc[common].values
        p = 1.0 if np.allclose(a, b) else stats.wilcoxon(a, b)[1]
        rows.append(dict(target=tgt, d=int(d), N=int(N), n=len(common),
                         kan_mean=round(float(a.mean()), 4), mlp_mean=round(float(b.mean()), 4),
                         median_diff=round(float(np.median(a - b)), 4),
                         rank_biserial=round(rank_biserial(a, b), 3),
                         p_value=round(float(p), 4)))
    sig = pd.DataFrame(rows)
    if len(sig):                                   # Benjamini-Hochberg
        p = sig.p_value.values.astype(float); m = len(p); order = np.argsort(p)
        q = np.empty(m); prev = 1.0
        for rank in range(m - 1, -1, -1):
            i = order[rank]; prev = min(prev, p[i] * m / (rank + 1)); q[i] = prev
        sig["q_value_BH"] = np.round(q, 4)
        sig["significant"] = np.where(sig.q_value_BH < THRESHOLDS["fdr_alpha"], "yes", "no")
    return sig


SAND = sandwich(runs)
P2, P2_TREND, AGG_CMP = absorption_gap(runs, CFG)
DSTAR = critical_dimension(AGG_CMP, CFG)
SIG = significance(runs)

print(f"P4 sandwich (Table 11): holds in {SAND.train_ok.mean()*100:.0f}% of {len(SAND)} cells; "
      f"mean test optimisation gap {SAND.opt_gap_test.mean():.4f}")
print("\nP2 absorption gap by alpha (Table 12):")
print(P2.groupby("alpha").absorption_gap.agg(["mean", "std", "size"]).round(3).to_string())
print("P2 trend test:", P2_TREND)
print("\nP3 critical dimension (Table 13):")
print(DSTAR.pivot(index="N", columns="alpha", values="d_star").to_string())
print(f"\nSignificance (Table 14): {(SIG.significant=='yes').sum()}/{len(SIG)} cells differ "
      f"at BH-FDR 5%; KAN lower error in {(SIG.median_diff<0).sum()} cells")
print(SIG.groupby("d").agg(cells=("n", "size"),
                           kan_better=("median_diff", lambda x: (x < 0).sum()),
                           mean_effect=("rank_biserial", "mean")).round(3).to_string())

## 12. Figures

Each figure is written to `outputs/figures/` as a vector PDF and a 300 dpi PNG. Titles are
left off the artwork; journals set them from the caption.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "serif", "font.size": 8, "axes.labelsize": 8, "legend.fontsize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.linewidth": 0.6,
    "lines.linewidth": 1.0, "lines.markersize": 3.5, "figure.dpi": 300,
})
W_HALF, W_FULL = 3.2, 6.5          # JMLR column and text widths, inches

def save_fig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), bbox_inches="tight",
                    dpi=300 if ext == "png" else None)
    plt.close(fig)
    print("written:", name)


# ---- Figure 1: bedrock rate ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(W_HALF, W_HALF * 0.85))
for k in CFG.spline_orders:
    errs = [np.mean([best_spline_error("additive", 1, CFG.n_train, CFG.n_test, s, G, k)[1]
                     for s in (0, 1, 2)]) for G in CFG.bedrock_grids]
    ax.plot(CFG.bedrock_grids, errs, "o-", label=f"$k={k}$")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel(r"spline grid $G$"); ax.set_ylabel(r"relative $L_2$ error"); ax.legend()
save_fig(fig, "fig_B0_bedrock_rate")


# ---- Figure 2: smoothness saturation ------------------------------------------------
fig, ax = plt.subplots(figsize=(W_HALF, W_HALF * 0.85))
for kk, g in SMOOTH_SAT.groupby("order_k"):
    g = g.sort_values("holder_H")
    ax.plot(g.holder_H, -g.measured_slope, "o-", label=f"$k={int(kk)}$")
H = np.linspace(SMOOTH_SAT.holder_H.min(), SMOOTH_SAT.holder_H.max(), 100)
for kk in sorted(SMOOTH_SAT.order_k.unique()):
    ax.plot(H, np.minimum(kk + 1, H), "--", alpha=0.4)
ax.set_xlabel(r"target H\"older smoothness $\mathcal{H}$")
ax.set_ylabel(r"effective rate"); ax.legend()
save_fig(fig, "fig_smoothness_saturation")


# ---- Figure 3: E/N collapse ---------------------------------------------------------
agg_probe = (probe_raw.groupby(["kind", "target", "k", "d", "grid", "params", "edges"],
                               dropna=False)["rel_l2"].mean().reset_index())
fig, axes = plt.subplots(1, 2, figsize=(W_FULL, W_FULL * 0.42))
for ax, kind, tgt, lab in [(axes[0], "edge_kan", "additive", "(a) additive, $E=d$"),
                           (axes[1], "kanet", "kastruct", r"(b) K--A, $E\sim 2d^2$")]:
    sub = agg_probe[(agg_probe.kind == kind) & (agg_probe.target == tgt)]
    for k in sorted(sub.k.dropna().unique()):
        gk = sub[(sub.k == k) & (sub.rel_l2 < SAT_DEFAULT) & (sub.edges > 0)]
        if len(gk) >= 2:
            ax.plot(gk.edges / gk.params, gk.rel_l2, "o", alpha=0.7, label=f"$k={int(k)}$")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(r"$E/N$"); ax.set_ylabel(r"relative $L_2$ error"); ax.legend()
    ax.text(-0.16, 1.02, lab, transform=ax.transAxes, fontweight="bold", va="bottom")
fig.tight_layout()
save_fig(fig, "fig_P1_EN_collapse")


# ---- Figure 4: critical dimension ---------------------------------------------------
NBIG = CFG.param_budgets[-1]
fig, axes = plt.subplots(2, 3, figsize=(W_FULL, W_FULL * 0.62), sharex=True, sharey=True)
axes = axes.ravel()
for j, al in enumerate(CFG.alpha_grid):
    ax = axes[j]
    for kind, style, lab, col in [("full_kan", "s-", "KAN", "#1f77b4"),
                                  ("mlp", "o--", "MLP", "#d62728")]:
        g = AGG_CMP[(AGG_CMP.kind == kind) & (AGG_CMP.alpha == al) & (AGG_CMP.N == NBIG)]
        if len(g):
            gg = g.groupby("d")["rel_l2"].min().reset_index().sort_values("d")
            ax.plot(gg.d, gg.rel_l2, style, color=col, label=lab, markerfacecolor="none")
    row = DSTAR[(DSTAR.N == NBIG) & (DSTAR.alpha == al)]
    if len(row):
        try:
            v = float(row.d_star.iloc[0])
            ax.axvline(v, color="0.45", lw=0.7, ls=":")
            ax.annotate(rf"$d^\ast\!=\!{v:g}$", xy=(v, 0.60),
                        xycoords=("data", "axes fraction"), xytext=(3, 0),
                        textcoords="offset points", fontsize=6.5, color="0.30",
                        rotation=90, va="center")
        except (TypeError, ValueError):
            ax.annotate("no crossing", xy=(0.06, 0.90), xycoords="axes fraction",
                        fontsize=6.5, color="0.30")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xticks([2, 4, 8, 16, 32])
    ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
    ax.text(0.03, 0.04, rf"({chr(97+j)}) $\alpha={al:g}$", transform=ax.transAxes,
            fontsize=7.5, fontweight="bold", va="bottom")
    ax.grid(True, lw=0.3, alpha=0.35)
axes[-1].axis("off")
h, l = axes[0].get_legend_handles_labels()
axes[-1].legend(h, l, loc="center", frameon=False, fontsize=8,
                title=r"best over $(k,\eta_0)$ per seed", title_fontsize=7.5)
for ax in (axes[0], axes[3]): ax.set_ylabel(r"relative $L_2$ error")
for ax in (axes[2], axes[3], axes[4]):
    ax.set_xlabel(r"input dimension $d$"); ax.tick_params(labelbottom=True)
fig.tight_layout(pad=0.4)
save_fig(fig, "fig_P3_critical_dim")

## 13. Pre-registered verdict

Each prediction is compared against the constants fixed in Section 1. No threshold is
modified here; where a prediction fails, it is recorded as failing.

In [ ]:
V = []
supp = lambda ok: "SUPPORTED" if ok else "UNSUPPORTED"

V.append(f"B0 bedrock rate [{supp(bool(BEDROCK.passes.all()))}]: slopes "
         + ", ".join(f"{r.measured_slope:.2f} (pred {r.predicted})"
                     for r in BEDROCK.itertuples()))

V.append(f"Reduction premise, single edge [{supp(bool(SINGLE_EDGE.passes.all()))}]: slopes "
         + ", ".join(f"{r.single_edge_slope:.2f}" for r in SINGLE_EDGE.itertuples())
         + f"; edge-count elasticity {DLOGE:.3f}, so the analytic d-slope is "
           f"{', '.join(f'{r.finite_d_slope}' for r in REDUCTION.itertuples())} at these dims")

for fam in JOINT.family.unique():
    sub = JOINT[JOINT.family == fam]
    V.append(f"{fam} -- asymptotic form [{supp(bool(sub.cE_pass.all()))}]: "
             + "; ".join(f"k={r.k}: c_E {r.cE:+.3f} [{r.cE_lo:+.3f},{r.cE_hi:+.3f}]"
                         for r in sub.itertuples())
             + f" | exact form [{supp(bool(sub.cG_pass.all()))}]: "
             + "; ".join(f"k={r.k}: c_G {r.cG:+.3f}" for r in sub.itertuples()))

V.append(f"Edge-law functional form [{supp(bool((NESTED[NESTED.family.str.startswith('best_spline')].F < 1).all()))}]: "
         f"for the oracle, adding log E beyond log G_exact gives F = "
         + ", ".join(f"{r.F}" for r in NESTED[NESTED.family.str.startswith('best_spline')].itertuples())
         + " -- error depends on (N,E) only through per-edge resolution")

_dsl = DSLOPE.loc[[i for i in DSLOPE.index if i.startswith("best_spline")]]
V.append(f"Premise, direct test [{supp(bool(_dsl.abs().max().max() <= 0.05))}]: the oracle's "
         f"dimensional slope at fixed resolution ranges "
         f"{_dsl.min().min():.2f} to {_dsl.max().max():.2f}; the premise predicts 0")

_ok = bool(SAND.train_ok.mean() >= THRESHOLDS["sandwich_train_min"])
V.append(f"P4 sandwich [{supp(_ok)}]: trained >= LSQ on TRAIN in "
         f"{SAND.train_ok.mean()*100:.0f}% of {len(SAND)} cells "
         f"(required >= {THRESHOLDS['sandwich_train_min']*100:.0f}%); "
         f"mean test optimisation gap {SAND.opt_gap_test.mean():.3f}")

V.append(f"P2 structural absorption [{supp(P2_TREND['supported'])}]: Spearman rho = "
         f"{P2_TREND['spearman_rho']} (p = {P2_TREND['spearman_p']}), bootstrap slope CI "
         f"[{P2_TREND['slope_lo']}, {P2_TREND['slope_hi']}]")

_finite = [r for r in DSTAR.itertuples() if isinstance(r.d_star, (int, float))]
V.append(f"P3 critical dimension [{supp(len(_finite) > 0)}]: a finite d* exists in "
         f"{len(_finite)}/{len(DSTAR)} cells, falling with budget and with alpha")

_nonid = JOINT[~JOINT.identifiable]
V.append(f"N-E identifiability [{supp(len(_nonid) == 0)}]: "
         + ("all fits have VIF < 10" if len(_nonid) == 0
            else f"{len(_nonid)} fit(s) collinear; their c_N/c_E split is not trustworthy"))

tm = pd.to_numeric(runs.train_mse, errors="coerce").dropna()
V.append(f"Convergence: median final train MSE {tm.median():.4f} "
         + ("(well fit)" if tm.median() < THRESHOLDS["converge_mse"] else "(elevated)"))

V.append(f"KAN vs MLP: {(SIG.significant=='yes').sum()}/{len(SIG)} cells differ at BH-FDR 5%; "
         f"KAN attains lower error in {(SIG.median_diff<0).sum()} cells, all with "
         f"d <= {SIG[SIG.median_diff<0].d.max()}")

verdict = "\n\n".join("- " + v for v in V)
print(verdict)
with open(os.path.join(OUT_DIR, "verdict.txt"), "w") as f:
    f.write(verdict)

manifest = dict(torch=torch.__version__, numpy=np.__version__,
                python=platform.python_version(), platform=platform.platform(),
                device=str(DEVICE), analysis_only=ANALYSIS_ONLY,
                config=asdict(CFG), thresholds=THRESHOLDS, n_cells=int(len(runs)),
                sha256_16=hashlib.sha256(
                    pd.util.hash_pandas_object(runs, index=True).values.tobytes()
                ).hexdigest()[:16])
with open(os.path.join(OUT_DIR, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)
print("\nresult-table digest:", manifest["sha256_16"])
print("written:", sorted(os.listdir(OUT_DIR)))